# Construction of Datasets21–25: Progressive Feature Engineering

This notebook implements the progressive feature-engineering pipeline used to construct Datasets21–25 for the LTE and 5G NR classification experiments.

Starting with Dataset20, new predictors are introduced in five stages:

- Dataset21: nearest infrastructure distances;
- Dataset22: local infrastructure densities;
- Dataset23: environmental–infrastructure interactions;
- Dataset24: remote-sensing-derived features; and
- Dataset25: geographic-context features.

The final Dataset25 contains 4,985 grid cells and 35 predictors.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Data Preparation)
- Section 4.1.3 (Progressive Feature Engineering)
- Table 4.2 (Summary of Progressive Feature Engineering)
- Appendix C (Reproducibility and Predictor Definitions)

In [ ]:
# ============================================================
# Step 1. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Load and Validate Dataset20

Dataset20 is loaded as the starting point for progressive feature engineering.

The dataset is checked for unique grid identifiers, coordinate-reference-system consistency, and valid spatial geometries before additional predictors are created.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Data Preparation)
- Section 4.1.3 (Progressive Feature Engineering)

In [ ]:
# ============================================================
# Dataset21–25 Feature Engineering
# Cell 1. Load and Validate Dataset20
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd

base_dir = (
    "/content/drive/MyDrive/Dissertation/Experiments/"
    "최종실험(수정)/Dataset_Build"
)

dataset20_path = os.path.join(
    base_dir,
    "08_Final_Datasets",
    "dataset20_final.gpkg"
)

dataset20 = gpd.read_file(
    dataset20_path,
    layer="dataset20_final"
)

# Validate Dataset20
print("Dataset20 shape:", dataset20.shape)
print("Unique grid IDs:", dataset20["grid_id"].nunique())
print("Duplicate grid IDs:", dataset20["grid_id"].duplicated().sum())
print("CRS:", dataset20.crs)
print("Invalid geometries:", (~dataset20.geometry.is_valid).sum())

print("\nColumns:")
print(dataset20.columns.tolist())

display(dataset20.head())

Dataset20 shape: (4985, 29)
Unique grid IDs: 4985
Duplicate grid IDs: 0
CRS: EPSG:27700
Invalid geometries: 0

Columns:
['grid_id', 'lte_min_rsrp', 'lte_mean_rsrp', 'lte_median_rsrp', 'lte_point_count', 'lte_signal_class', 'nr_min_rsrp', 'nr_mean_rsrp', 'nr_median_rsrp', 'nr_point_count', 'nr_signal_class', 'VV', 'VH', 'NDVI', 'NDBI', 'population', 'nightlight', 'tree_ratio', 'grass_ratio', 'crop_ratio', 'builtup_ratio', 'water_ratio', 'cell_count', 'operator_count', 'gsm_count', 'umts_count', 'lte_count', 'nr_count', 'geometry']


,grid_id,lte_min_rsrp,lte_mean_rsrp,lte_median_rsrp,lte_point_count,lte_signal_class,nr_min_rsrp,nr_mean_rsrp,nr_median_rsrp,nr_point_count,...,crop_ratio,builtup_ratio,water_ratio,cell_count,operator_count,gsm_count,umts_count,lte_count,nr_count,geometry
0,3975,-55.64,-46.228794,-45.48,937.0,Excellent,-128.64,-89.648030,-92.37,1755.0,...,0.235360,0.197890,0.000000,5,2,0,0,5,0,"POLYGON ((426000 576000, 426000 577000, 425000..."
1,3976,-81.71,-68.405396,-69.13,1609.0,Excellent,-130.87,-92.905040,-92.43,2657.0,...,0.000000,0.522511,0.000000,0,0,0,0,0,0,"POLYGON ((427000 576000, 427000 577000, 426000..."
2,4072,-89.99,-83.527565,-84.71,1134.0,Good,-138.30,-100.509040,-96.83,2055.0,...,0.000000,0.421669,0.000000,0,0,0,0,0,0,"POLYGON ((427000 575000, 427000 576000, 426000..."
3,4166,-74.81,-64.096870,-62.38,624.0,Excellent,-128.37,-90.326965,-93.48,1207.0,...,0.067523,0.152720,0.000057,4,2,0,0,4,0,"POLYGON ((424000 574000, 424000 575000, 423000..."
4,4167,-74.29,-70.279630,-70.44,216.0,Excellent,-131.27,-91.655655,-94.24,361.0,...,0.424147,0.027295,0.000746,0,0,0,0,0,0,"POLYGON ((425000 574000, 425000 575000, 424000..."


## 2. Dataset21: Nearest Infrastructure Distance

OpenCellID records are cleaned, reprojected to the British National Grid, and separated into LTE and 5G NR records.

Spatial search trees are used to calculate the straight-line distance from each grid centroid to the nearest LTE and 5G NR OpenCellID records.

The following predictors are added:

- `nearest_lte_distance_m`
- `nearest_nr_distance_m`

### ※ Related dissertation sections

- Section 4.1.3 (Progressive Feature Engineering)
- Table 4.2 (Summary of Progressive Feature Engineering)
- Appendix C, Table C.2 (Definitions of the 35 Predictors)

In [ ]:
# ============================================================
# Dataset21 (Nearest Distance)
# Cell 2. Prepare Grid Centroids and OpenCellID Points
# ============================================================

# ------------------------------------------------------------
# 1. Prepare grid centroids
# ------------------------------------------------------------

grid_gdf = dataset20.copy()

grid_centroids = grid_gdf.geometry.centroid

grid_xy = np.column_stack([
    grid_centroids.x.to_numpy(),
    grid_centroids.y.to_numpy()
])

# ------------------------------------------------------------
# 2. Load OpenCellID records
# ------------------------------------------------------------

opencellid_path = (
    "/content/drive/MyDrive/Dissertation/"
    "OpencellID/234_raw.csv"
)

if not os.path.exists(opencellid_path):
    raise FileNotFoundError(
        f"OpenCellID file not found: {opencellid_path}"
    )

ocid_columns = [
    "radio", "mcc", "net", "area",
    "cell", "unit", "lon", "lat"
]

ocid = pd.read_csv(
    opencellid_path,
    usecols=ocid_columns,
    low_memory=False
)

# Remove missing coordinates and duplicate records
ocid = ocid.dropna(
    subset=["lon", "lat", "radio"]
).drop_duplicates(
    subset=[
        "radio", "mcc", "net",
        "area", "cell", "unit",
        "lon", "lat"
    ]
).copy()

# Convert records to British National Grid
ocid_gdf = gpd.GeoDataFrame(
    ocid,
    geometry=gpd.points_from_xy(
        ocid["lon"],
        ocid["lat"]
    ),
    crs="EPSG:4326"
).to_crs(epsg=27700)

# ------------------------------------------------------------
# 3. Separate LTE and 5G NR records
# ------------------------------------------------------------

lte_records = ocid_gdf[
    ocid_gdf["radio"] == "LTE"
].copy()

nr_records = ocid_gdf[
    ocid_gdf["radio"] == "NR"
].copy()

print("Dataset20 grids:", f"{len(grid_gdf):,}")
print("LTE records:", f"{len(lte_records):,}")
print("5G NR records:", f"{len(nr_records):,}")

print("\nGrid CRS:", grid_gdf.crs)
print("OpenCellID CRS:", ocid_gdf.crs)

Dataset20 grids: 4,985
LTE records: 110,030
5G NR records: 4,188

Grid CRS: EPSG:27700
OpenCellID CRS: EPSG:27700


In [ ]:
# ============================================================
# Dataset21 (Nearest Distance)
# Cell 3. Calculate Nearest LTE and 5G NR Record Distances
# ============================================================

from scipy.spatial import cKDTree

# Prepare LTE and 5G NR record coordinates
lte_xy = np.column_stack([
    lte_records.geometry.x.to_numpy(),
    lte_records.geometry.y.to_numpy()
])

nr_xy = np.column_stack([
    nr_records.geometry.x.to_numpy(),
    nr_records.geometry.y.to_numpy()
])

if len(lte_xy) == 0 or len(nr_xy) == 0:
    raise ValueError("LTE or 5G NR coordinate data is empty.")

# Build spatial search trees
lte_tree = cKDTree(lte_xy)
nr_tree = cKDTree(nr_xy)

# Find the nearest record to each grid centroid
nearest_lte_distance, _ = lte_tree.query(
    grid_xy,
    k=1
)

nearest_nr_distance, _ = nr_tree.query(
    grid_xy,
    k=1
)

# Create Dataset21
dataset21 = dataset20.copy()

dataset21["nearest_lte_distance_m"] = (
    nearest_lte_distance
)

dataset21["nearest_nr_distance_m"] = (
    nearest_nr_distance
)

# Validate Dataset21
if len(dataset21) != len(dataset20):
    raise ValueError("Dataset21 row count does not match Dataset20.")

if not np.isfinite(
    dataset21[
        [
            "nearest_lte_distance_m",
            "nearest_nr_distance_m"
        ]
    ].to_numpy()
).all():
    raise ValueError("Invalid nearest-distance values were found.")

print("Dataset21 shape:", dataset21.shape)
print("Unique grid IDs:", dataset21["grid_id"].nunique())

print("\nNearest LTE record distance (m):")
print(dataset21["nearest_lte_distance_m"].describe())

print("\nNearest 5G NR record distance (m):")
print(dataset21["nearest_nr_distance_m"].describe())

Dataset21 shape: (4985, 31)
Unique grid IDs: 4985

Nearest LTE record distance (m):
count     4985.000000
mean       800.756624
std       1630.952742
min          4.795423
25%        182.574626
50%        360.266946
75%        666.125579
max      16855.320002
Name: nearest_lte_distance_m, dtype: float64

Nearest 5G NR record distance (m):
count      4985.000000
mean      17105.854853
std       29108.010358
min          21.654604
25%        2222.733372
50%        5719.779058
75%       14223.968926
max      129482.103591
Name: nearest_nr_distance_m, dtype: float64


In [ ]:
# ============================================================
# Dataset21 (Nearest Distance)
# Cell 4. Save Dataset21
# ============================================================

dataset21_output_dir = os.path.join(
    base_dir,
    "09_Feature_Engineering",
    "Dataset21"
)

os.makedirs(
    dataset21_output_dir,
    exist_ok=True
)

csv_path = os.path.join(
    dataset21_output_dir,
    "dataset21_final.csv"
)

gpkg_path = os.path.join(
    dataset21_output_dir,
    "dataset21_final.gpkg"
)

# Save tabular data without geometry
dataset21.drop(
    columns="geometry"
).to_csv(
    csv_path,
    index=False
)

# Save spatial data with geometry
dataset21.to_file(
    gpkg_path,
    layer="dataset21_final",
    driver="GPKG"
)

print("Dataset21 shape:", dataset21.shape)

print("\nCSV saved:")
print(csv_path)

print("\nGeoPackage saved:")
print(gpkg_path)

Dataset21 shape: (4985, 31)

CSV saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/09_Feature_Engineering/Dataset21/dataset21_final.csv

GeoPackage saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/09_Feature_Engineering/Dataset21/dataset21_final.gpkg


## 3. Dataset22: Local Infrastructure Density

Local cellular-infrastructure context is represented by counting LTE and 5G NR OpenCellID records within 3 km and 5 km of each grid centroid.

The following predictors are added:

- 'lte_density_3km'
- 'nr_density_3km'
- 'lte_density_5km'
- 'nr_density_5km'

The 5 km counts are validated to ensure that they are not lower than the corresponding 3 km counts.

### ※ Related dissertation sections

- Section 4.1.3 (Progressive Feature Engineering)
- Table 4.2 (Summary of Progressive Feature Engineering)
- Appendix C, Table C.2 (Definitions of the 35 Predictors)

In [ ]:
# ============================================================
# Dataset22 (Local Density)
# Cell 5. Calculate LTE and 5G NR Record Counts within 3 km and 5 km
# ============================================================

# Count nearby OpenCellID records using spatial search trees
lte_density_3km = lte_tree.query_ball_point(
    grid_xy,
    r=3000,
    return_length=True
)

nr_density_3km = nr_tree.query_ball_point(
    grid_xy,
    r=3000,
    return_length=True
)

lte_density_5km = lte_tree.query_ball_point(
    grid_xy,
    r=5000,
    return_length=True
)

nr_density_5km = nr_tree.query_ball_point(
    grid_xy,
    r=5000,
    return_length=True
)

# Create Dataset22
dataset22 = dataset21.copy()

dataset22["lte_density_3km"] = (
    lte_density_3km.astype("int32")
)

dataset22["nr_density_3km"] = (
    nr_density_3km.astype("int32")
)

dataset22["lte_density_5km"] = (
    lte_density_5km.astype("int32")
)

dataset22["nr_density_5km"] = (
    nr_density_5km.astype("int32")
)

density_cols = [
    "lte_density_3km",
    "nr_density_3km",
    "lte_density_5km",
    "nr_density_5km"
]

# Validate Dataset22
if len(dataset22) != len(dataset21):
    raise ValueError("Dataset22 row count does not match Dataset21.")

if not (
    dataset22["lte_density_5km"]
    >= dataset22["lte_density_3km"]
).all():
    raise ValueError("Invalid LTE density counts were found.")

if not (
    dataset22["nr_density_5km"]
    >= dataset22["nr_density_3km"]
).all():
    raise ValueError("Invalid 5G NR density counts were found.")

print("Dataset22 shape:", dataset22.shape)
print("Unique grid IDs:", dataset22["grid_id"].nunique())

print("\nLocal infrastructure summary:")
display(dataset22[density_cols].describe())

Dataset22 shape: (4985, 35)
Unique grid IDs: 4985

Local infrastructure summary:


,lte_density_3km,nr_density_3km,lte_density_5km,nr_density_5km
count,4985.000000,4985.000000,4985.000000,4985.000000
mean,114.718556,4.359077,289.383350,11.023470
std,143.914972,10.585898,334.993638,20.465089
min,0.000000,0.000000,0.000000,0.000000
25%,24.000000,0.000000,63.000000,0.000000
50%,67.000000,0.000000,171.000000,0.000000
75%,151.000000,2.000000,387.000000,13.000000
max,1352.000000,91.000000,2444.000000,160.000000


In [ ]:
# ============================================================
# Dataset22 (Local Density)
# Cell 6. Save Dataset22
# ============================================================

dataset22_output_dir = os.path.join(
    base_dir,
    "09_Feature_Engineering",
    "Dataset22"
)

os.makedirs(
    dataset22_output_dir,
    exist_ok=True
)

csv_path = os.path.join(
    dataset22_output_dir,
    "dataset22_final.csv"
)

gpkg_path = os.path.join(
    dataset22_output_dir,
    "dataset22_final.gpkg"
)

# Save tabular data without geometry
dataset22.drop(
    columns="geometry"
).to_csv(
    csv_path,
    index=False
)

# Save spatial data with geometry
dataset22.to_file(
    gpkg_path,
    layer="dataset22_final",
    driver="GPKG"
)

print("Dataset22 shape:", dataset22.shape)

print("\nCSV saved:")
print(csv_path)

print("\nGeoPackage saved:")
print(gpkg_path)

Dataset22 shape: (4985, 35)

CSV saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/09_Feature_Engineering/Dataset22/dataset22_final.csv

GeoPackage saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/09_Feature_Engineering/Dataset22/dataset22_final.gpkg


## 4. Dataset23: Environmental–Infrastructure Interactions

Interaction features are created to represent how cellular-infrastructure availability may combine with population, built-up land, and night-time-light conditions.

The following predictors are added:

- population × LTE and 5G NR record counts;
- built-up ratio × LTE and 5G NR record counts; and
- night-time light × LTE and 5G NR record counts.

These features were motivated by the combined patterns identified during exploratory data analysis.

### ※ Related dissertation sections

- Section 4.1.2 (Exploratory Data Analysis)
- Section 4.1.3 (Progressive Feature Engineering)
- Table 4.1 (Summary of EDA Findings and Feature-Engineering Responses)
- Table 4.2 (Summary of Progressive Feature Engineering)

In [ ]:
# ============================================================
# Dataset23 (Interaction Features)
# Cell 7. Create and Validate Interaction Features
# ============================================================

dataset23 = dataset22.copy()

# Population × infrastructure
dataset23["pop_x_lte"] = (
    dataset23["population"]
    * dataset23["lte_count"]
)

dataset23["pop_x_nr"] = (
    dataset23["population"]
    * dataset23["nr_count"]
)

# Built-up ratio × infrastructure
dataset23["builtup_x_lte"] = (
    dataset23["builtup_ratio"]
    * dataset23["lte_count"]
)

dataset23["builtup_x_nr"] = (
    dataset23["builtup_ratio"]
    * dataset23["nr_count"]
)

# Nightlight × infrastructure
dataset23["nightlight_x_lte"] = (
    dataset23["nightlight"]
    * dataset23["lte_count"]
)

dataset23["nightlight_x_nr"] = (
    dataset23["nightlight"]
    * dataset23["nr_count"]
)

interaction_cols = [
    "pop_x_lte",
    "pop_x_nr",
    "builtup_x_lte",
    "builtup_x_nr",
    "nightlight_x_lte",
    "nightlight_x_nr"
]

# Validate interaction features
if dataset23[interaction_cols].isna().any().any():
    raise ValueError("Missing interaction-feature values were found.")

if not np.isfinite(
    dataset23[interaction_cols].to_numpy()
).all():
    raise ValueError("Infinite interaction-feature values were found.")

if len(dataset23) != len(dataset22):
    raise ValueError("Dataset23 row count does not match Dataset22.")

print("Dataset23 shape:", dataset23.shape)
print("Unique grid IDs:", dataset23["grid_id"].nunique())

print("\nInteraction-feature summary:")
display(dataset23[interaction_cols].describe())

Dataset23 shape: (4985, 41)
Unique grid IDs: 4985

Interaction-feature summary:


,pop_x_lte,pop_x_nr,builtup_x_lte,builtup_x_nr,nightlight_x_lte,nightlight_x_nr
count,4.985000e+03,4985.000000,4985.000000,4985.000000,4985.000000,4985.000000
mean,1.259031e+04,510.246273,2.184206,0.094382,125.129039,5.049073
std,4.339583e+04,4560.852823,6.802560,0.881977,504.395469,54.905751
min,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
50%,4.774242e+02,0.000000,0.311404,0.000000,12.864137,0.000000
75%,7.509855e+03,0.000000,1.736636,0.000000,82.252257,0.000000
max,1.352530e+06,170866.197950,168.590951,41.164290,13599.495463,2740.248062


In [ ]:
# ============================================================
# Dataset23 (Interaction Features)
# Cell 8. Save Dataset23
# ============================================================

dataset23_output_dir = os.path.join(
    base_dir,
    "09_Feature_Engineering",
    "Dataset23"
)

os.makedirs(
    dataset23_output_dir,
    exist_ok=True
)

csv_path = os.path.join(
    dataset23_output_dir,
    "dataset23_final.csv"
)

gpkg_path = os.path.join(
    dataset23_output_dir,
    "dataset23_final.gpkg"
)

# Save tabular data without geometry
dataset23.drop(
    columns="geometry"
).to_csv(
    csv_path,
    index=False
)

# Save spatial data with geometry
dataset23.to_file(
    gpkg_path,
    layer="dataset23_final",
    driver="GPKG"
)

print("Dataset23 shape:", dataset23.shape)
print("Unique grid IDs:", dataset23["grid_id"].nunique())

print("\nCSV saved:")
print(csv_path)

print("\nGeoPackage saved:")
print(gpkg_path)

Dataset23 shape: (4985, 41)
Unique grid IDs: 4985

CSV saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/09_Feature_Engineering/Dataset23/dataset23_final.csv

GeoPackage saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/09_Feature_Engineering/Dataset23/dataset23_final.gpkg


## 5. Dataset24: Remote-Sensing-Derived Features

Three derived remote-sensing predictors are created to represent combined surface and environmental characteristics:

- 'vv_x_ndvi': Sentinel-1 VV × NDVI;
- 'vh_x_builtup': Sentinel-1 VH × built-up ratio; and
- 'ndvi_minus_ndbi': NDVI − NDBI.

The resulting values are checked for missing and non-finite observations.

### ※ Related dissertation sections

- Section 2.6 (Public Geospatial Data as Predictors)
- Section 4.1.2 (Exploratory Data Analysis)
- Section 4.1.3 (Progressive Feature Engineering)
- Table 4.2 (Summary of Progressive Feature Engineering)

In [ ]:
# ============================================================
# Dataset24 (Remote-Sensing Interaction Features)
# Cell 9. Create and Validate Remote-Sensing Features
# ============================================================

dataset24 = dataset23.copy()

# Create remote-sensing interaction features
dataset24["vv_x_ndvi"] = (
    dataset24["VV"]
    * dataset24["NDVI"]
)

dataset24["vh_x_builtup"] = (
    dataset24["VH"]
    * dataset24["builtup_ratio"]
)

dataset24["ndvi_minus_ndbi"] = (
    dataset24["NDVI"]
    - dataset24["NDBI"]
)

rs_interaction_cols = [
    "vv_x_ndvi",
    "vh_x_builtup",
    "ndvi_minus_ndbi"
]

# Validate derived features
if dataset24[rs_interaction_cols].isna().any().any():
    raise ValueError(
        "Missing remote-sensing interaction values were found."
    )

if not np.isfinite(
    dataset24[rs_interaction_cols].to_numpy()
).all():
    raise ValueError(
        "Infinite remote-sensing interaction values were found."
    )

if len(dataset24) != len(dataset23):
    raise ValueError(
        "Dataset24 row count does not match Dataset23."
    )

print("Dataset24 shape:", dataset24.shape)
print("Unique grid IDs:", dataset24["grid_id"].nunique())

print("\nRemote-sensing feature summary:")
display(dataset24[rs_interaction_cols].describe())

Dataset24 shape: (4985, 44)
Unique grid IDs: 4985

Remote-sensing feature summary:


,vv_x_ndvi,vh_x_builtup,ndvi_minus_ndbi
count,4985.000000,4985.000000,4985.000000
mean,-5.337446,-3.945369,0.659480
std,2.115640,3.272200,0.207884
min,-11.528123,-12.327044,0.034724
25%,-6.869372,-6.779903,0.515264
50%,-5.184311,-3.007724,0.650838
75%,-3.786057,-0.960397,0.806938
max,2.539465,-0.000000,1.230788


In [ ]:
# ============================================================
# Dataset24 (Remote-Sensing Interaction Features)
# Cell 10. Save Dataset24
# ============================================================

dataset24_output_dir = os.path.join(
    base_dir,
    "09_Feature_Engineering",
    "Dataset24"
)

os.makedirs(
    dataset24_output_dir,
    exist_ok=True
)

csv_path = os.path.join(
    dataset24_output_dir,
    "dataset24_final.csv"
)

gpkg_path = os.path.join(
    dataset24_output_dir,
    "dataset24_final.gpkg"
)

# Save tabular data without geometry
dataset24.drop(
    columns="geometry"
).to_csv(
    csv_path,
    index=False
)

# Save spatial data with geometry
dataset24.to_file(
    gpkg_path,
    layer="dataset24_final",
    driver="GPKG"
)

print("Dataset24 shape:", dataset24.shape)
print("Unique grid IDs:", dataset24["grid_id"].nunique())

print("\nCSV saved:")
print(csv_path)

print("\nGeoPackage saved:")
print(gpkg_path)

Dataset24 shape: (4985, 44)
Unique grid IDs: 4985

CSV saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/09_Feature_Engineering/Dataset24/dataset24_final.csv

GeoPackage saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/09_Feature_Engineering/Dataset24/dataset24_final.gpkg


## 6. Dataset25: Geographic Context

Three geographic-context predictors are calculated from the grid centroids:

- longitude;
- latitude; and
- straight-line distance to central London.

Longitude and latitude describe broad spatial location, while distance from London provides an interpretable measure of geographic centrality.

### ※ Related dissertation sections

- Section 4.1.2 (Exploratory Data Analysis)
- Section 4.1.3 (Progressive Feature Engineering)
- Table 4.2 (Summary of Progressive Feature Engineering)
- Appendix C, Table C.2 (Definitions of the 35 Predictors)

In [ ]:
# ============================================================
# Dataset25 (Geospatial Context)
# Cell 11. Create Geographic Features
# ============================================================

from shapely.geometry import Point

# Start from Dataset24
dataset25 = dataset24.copy()

# Calculate grid centroids in British National Grid
centroids_bng = dataset25.geometry.centroid

# Convert centroids to longitude and latitude
centroids_wgs84 = centroids_bng.to_crs(
    epsg=4326
)

dataset25["longitude"] = (
    centroids_wgs84.x.to_numpy()
)

dataset25["latitude"] = (
    centroids_wgs84.y.to_numpy()
)

# Create London reference point
london_bng = gpd.GeoSeries(
    [Point(-0.1278, 51.5074)],
    crs="EPSG:4326"
).to_crs(
    epsg=27700
).iloc[0]

# Calculate straight-line distance to London
dataset25["distance_to_london_km"] = (
    centroids_bng.distance(london_bng)
    / 1000
)

geo_cols = [
    "longitude",
    "latitude",
    "distance_to_london_km"
]

# Validate geographic features
if dataset25[geo_cols].isna().any().any():
    raise ValueError("Missing geographic values were found.")

if not np.isfinite(
    dataset25[geo_cols].to_numpy()
).all():
    raise ValueError("Invalid geographic values were found.")

if len(dataset25) != len(dataset24):
    raise ValueError("Dataset25 row count does not match Dataset24.")

print("Dataset25 shape:", dataset25.shape)
print("Unique grid IDs:", dataset25["grid_id"].nunique())

print("\nGeographic-feature summary:")
display(dataset25[geo_cols].describe())

print("\nFirst five rows:")
display(
    dataset25[
        ["grid_id"] + geo_cols
    ].head()
)

Dataset25 shape: (4985, 47)
Unique grid IDs: 4985

Geographic-feature summary:


,longitude,latitude,distance_to_london_km
count,4985.000000,4985.000000,4985.000000
mean,-1.444906,52.998756,198.094034
std,0.774273,1.007973,107.999186
min,-3.056898,51.297862,9.660103
25%,-1.994058,52.065439,132.440182
50%,-1.523877,52.997427,212.070569
75%,-0.903027,53.779056,272.649561
max,0.231333,55.082296,435.493590



First five rows:


,grid_id,longitude,latitude,distance_to_london_km
0,3975,-1.602096,55.082296,409.679434
1,3976,-1.586432,55.082244,409.425428
2,4072,-1.586525,55.073258,408.458004
3,4166,-1.633588,55.064423,408.263241
4,4167,-1.617931,55.064375,408.003452


## 7. Final Dataset Validation and Export

Dataset25 is validated to confirm:

- 4,985 unique grid cells
- 35 predictor variables
- no missing predictor values
- no duplicate grid identifiers and
- preservation of the LTE and 5G NR labelled subsets.

Target-generating RSRP statistics and measurement counts are retained for analysis but identified separately from the predictor matrix to prevent target leakage.

The final dataset is exported in CSV and GeoPackage formats.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Data Preparation)
- Section 4.1.3 (Progressive Feature Engineering)
- Section 4.2 (Effect of Progressive Feature Engineering on Classification Performance)
- Appendix C (Reproducibility and Predictor Definitions)

In [ ]:
# ============================================================
# Dataset25 (Geospatial Context)
# Cell 12. Validate and Save Final Dataset25
# ============================================================

# Target-related columns that must not be used as predictors
target_cols = [
    "lte_min_rsrp",
    "lte_mean_rsrp",
    "lte_median_rsrp",
    "lte_point_count",
    "lte_signal_class",
    "nr_min_rsrp",
    "nr_mean_rsrp",
    "nr_median_rsrp",
    "nr_point_count",
    "nr_signal_class"
]

# Identify the 35 predictor columns
predictor_cols = [
    column
    for column in dataset25.columns
    if column not in target_cols
    and column not in ["grid_id", "geometry"]
]

# Validate final dataset
if len(dataset25) != 4985:
    raise ValueError("Unexpected Dataset25 row count.")

if dataset25["grid_id"].duplicated().any():
    raise ValueError("Duplicate grid IDs were found.")

if dataset25[predictor_cols].isna().any().any():
    raise ValueError("Missing predictor values were found.")

if len(predictor_cols) != 35:
    raise ValueError(
        f"Expected 35 predictors, but found {len(predictor_cols)}."
    )

print("Dataset25 shape:", dataset25.shape)
print("Unique grid IDs:", dataset25["grid_id"].nunique())
print("Number of predictors:", len(predictor_cols))

print("\nLTE labelled grids:")
print(dataset25["lte_signal_class"].notna().sum())

print("\n5G NR labelled grids:")
print(dataset25["nr_signal_class"].notna().sum())

print("\nMissing predictor values:")
print(dataset25[predictor_cols].isna().sum().sum())

# Define final output directory
output_dir = os.path.join(
    base_dir,
    "08_Final_Datasets"
)

csv_path = os.path.join(
    output_dir,
    "dataset25_final.csv"
)

gpkg_path = os.path.join(
    output_dir,
    "dataset25_final.gpkg"
)

# Save tabular data without geometry
dataset25.drop(
    columns="geometry"
).to_csv(
    csv_path,
    index=False
)

# Save spatial data with geometry
dataset25.to_file(
    gpkg_path,
    layer="dataset25_final",
    driver="GPKG"
)

print("\nCSV saved:")
print(csv_path)

print("\nGeoPackage saved:")
print(gpkg_path)

Dataset25 shape: (4985, 47)
Unique grid IDs: 4985
Number of predictors: 35

LTE labelled grids:
4970

5G NR labelled grids:
4965

Missing predictor values:
0

CSV saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/08_Final_Datasets/dataset25_final.csv

GeoPackage saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/08_Final_Datasets/dataset25_final.gpkg
